# Device Control Service (DCS) - Hue Lamp

This notebook demonstrates the Device Control Service for the Hue Lamp. The Hue Lamp is a smart light bulb that can be controlled using its Thing Description. The Device Control Service maps The gesture events to the actions in the Thing Description to control the Hue Lamp.

## Import Libraries


In [1]:
import time

import requests
import zmq
import msgpack
from pprint import pprint

# HueLamp Class

The `HueLamp` class provides methods to control the Hue Lamp. The class uses the Thing Description URL of the Hue Lamp to get and update the lamp state. The class provides methods to set the brightness, color, and power state of the lamp.

In [2]:
class HueLamp:
    def __init__(self, td_url):
        self.td_url = td_url

    def _get_lamp_state(self):
        try:
            response = requests.get(self.td_url)
            response.raise_for_status()
            return response.json()
        except requests.RequestException as e:
            print(f"Error retrieving lamp state: {e}")
            return None

    def _update_lamp_state(self, payload):
        try:
            response = requests.put(self.td_url, json=payload)
            response.raise_for_status()
            return response.status_code == 200
        except requests.RequestException as e:
            print(f"Error updating lamp state: {e}")
            return False

    def set_brightness(self, mode):
        state = self._get_lamp_state()
        if not state or not state.get("on"):
            self.toggle_power(True)
            return

        brightness = state.get("brightness", 0)
        if mode == 1:
            brightness = min(brightness + 20, 100)
        elif mode == -1:
            brightness = max(brightness - 20, 0)

        if self._update_lamp_state({"brightness": brightness}):
            print(f"Lamp brightness set to {brightness}.")

    def set_color(self, mode):
        state = self._get_lamp_state()
        if not state or not state.get("on"):
            self.toggle_power(True)
            return

        colors = ["red", "green", "gold", "violet"]
        curr_color = state.get("color", "red")
        curr_index = colors.index(curr_color) if curr_color in colors else 0

        if mode == -1:
            curr_index = (curr_index - 1) % len(colors)
        elif mode == 1:
            curr_index = (curr_index + 1) % len(colors)

        new_color = colors[curr_index]
        if self._update_lamp_state({"color": new_color}):
            print(f"Lamp color set to {new_color}.")

    def toggle_power(self, state):
        if self._update_lamp_state({"on": state}):
            print(f"Lamp turned {'on' if state else 'off'}.")

# TractorBot Class

The `TractorBot` class provides methods to control the TractorBot. The class uses the action URL of the TractorBot to send commands for movement. The class provides methods to move the TractorBot in different directions.

In [3]:
class TractorBot:
    def __init__(self, action_url):
        self.action_url = action_url

    def move(self, payload):
        if not payload:
            print(f"Invalid mode: {mode}")
            return

        try:
            response = requests.post(self.action_url, json=payload, headers={"Content-Type": "application/json"})
            response.raise_for_status()
            print(f"TractorBot action succeeded: {response.json()}")
        except requests.RequestException as e:
            print(f"Error moving TractorBot: {e}")

## Create Socket

In [4]:
def create_socket(ctx, ip, port, topics):
    sub = ctx.socket(zmq.SUB)
    sub.connect(f'tcp://{ip}:{port}')
    for topic in topics:
        sub.subscribe(topic)
    return sub

## Mapping of Gestures to Actions

In [13]:
TRACTOR_BOT_MAPPING = {
    0: {
        "right": {"axis": 0, "speed": 2, "duration": 1000},
        "left": {"axis": 0, "speed": -2, "duration": 1000},
        "up": {"axis": 2, "speed": -3, "duration": 500},
        "down": {"axis": 2, "speed": 3, "duration": 500}
    },
    90: {
        "up": {"axis": 0, "speed": 2, "duration": 1000},
        "down": {"axis": 0, "speed": -2, "duration": 1000},
        "left": {"axis": 2, "speed": -3, "duration": 500},
        "right": {"axis": 2, "speed": 3, "duration": 500}
    },
    180: {
        "left": {"axis": 0, "speed": 2, "duration": 1000},
        "right": {"axis": 0, "speed": -2, "duration": 1000},
        "down": {"axis": 2, "speed": -3, "duration": 500},
        "up": {"axis": 2, "speed": 3, "duration": 500}
    },
    270: {
        "down": {"axis": 0, "speed": 2, "duration": 1000},
        "up": {"axis": 0, "speed": -2, "duration": 1000},
        "right": {"axis": 2, "speed": -3, "duration": 500},
        "left": {"axis": 2, "speed": 3, "duration": 500}
    }
}


HUE_LAMP_MAPPING = {
    "left": ("color", -1),
    "right": ("color", 1),
    "up": ("brightness", 1),
    "down": ("brightness", -1),
}

## Specify Gesture Type

In [14]:
HEAD_GESTURE_TOPIC = "head_gesture"
BLINK_GESTURE_TOPIC = "blink_gestures"
SACCADE_GESTURE_TOPIC = "saccade_gestures"

topics = [HEAD_GESTURE_TOPIC, SACCADE_GESTURE_TOPIC]

## Start the Device Control Service

In [15]:
lamp = HueLamp(td_url="http://10.2.2.33:1880/r402/theglobe")
tractor_bot = TractorBot(action_url="http://10.2.2.183/actions/wheelcontrol")

curr_alignment = 0

ctx = zmq.Context()
socket = ctx.socket(zmq.REQ)
ip = 'localhost'
port = 50020

socket.connect(f'tcp://{ip}:{port}')
socket.send_string('SUB_PORT')
sub_port = socket.recv_string()

socket.send_string('PUB_PORT')
pub_port = socket.recv_string()
socket.close()

while True:
    socket_sub = create_socket(ctx, ip, sub_port, topics)
    topic = socket_sub.recv_string()
    payload = socket_sub.recv()
    msg = msgpack.unpackb(payload, raw=False)
    socket_sub.close()
    pprint(msg)

    if "HueLamp" in msg.get("object", ""):
        mode = msg["direction"]
        if mode in HUE_LAMP_MAPPING:
            attr, value = HUE_LAMP_MAPPING[mode]
            getattr(lamp, f"set_{attr}")(value)

    elif "Tractorbot" in msg.get("object", ""):
        mode = msg["direction"]

        if mode in TRACTOR_BOT_MAPPING[curr_alignment]:
            payload = TRACTOR_BOT_MAPPING[curr_alignment][mode]
            tractor_bot.move(payload)

        if curr_alignment == 0 and mode == "up":
            curr_alignment = 90
        elif curr_alignment == 0 and mode == "down":
            curr_alignment = 270
        elif curr_alignment == 90 and mode == "left":
            curr_alignment = 180
        elif curr_alignment == 90 and mode == "right":
            curr_alignment = 0
        elif curr_alignment == 180 and mode == "down":
            curr_alignment = 270
        elif curr_alignment == 180 and mode == "up":
            curr_alignment = 90
        elif curr_alignment == 270 and mode == "right":
            curr_alignment = 0
        elif curr_alignment == 270 and mode == "left":
            curr_alignment = 180




{'direction': 'right',
 'object': 'Tractorbot',
 'timestamp': 830.989287973,
 'topic': 'head_gestures'}
TractorBot action succeeded: {'axis': 0, 'speed': 2, 'duration': 1000}
{'direction': 'left',
 'object': 'Tractorbot',
 'timestamp': 835.245491106,
 'topic': 'head_gestures'}
TractorBot action succeeded: {'axis': 0, 'speed': -2, 'duration': 1000}
{'direction': 'down',
 'object': 'Tractorbot',
 'timestamp': 837.866621558,
 'topic': 'head_gestures'}
TractorBot action succeeded: {'axis': 2, 'speed': 3, 'duration': 500}
{'direction': 'up',
 'object': 'Tractorbot',
 'timestamp': 844.869995918,
 'topic': 'head_gestures'}
TractorBot action succeeded: {'axis': 0, 'speed': -2, 'duration': 1000}
{'direction': 'down',
 'object': 'Tractorbot',
 'timestamp': 859.241212821,
 'topic': 'head_gestures'}
TractorBot action succeeded: {'axis': 0, 'speed': 2, 'duration': 1000}
{'direction': 'down',
 'object': 'Tractorbot',
 'timestamp': 864.883689314,
 'topic': 'head_gestures'}
TractorBot action succeeded

KeyboardInterrupt: 